# FM Grand Prix · Servidor en Colab Pro (para probar desde tu VS Code)

Este notebook levanta **`Qwen/Qwen2.5-0.5B`** en la GPU de tu **Colab Pro** y abre un túnel público HTTPS para que desde tu **VS Code local** ejecutes:
```bash
python option1/entrenar_y_evaluar.py --enviar-gcp
```
y pruebes en vivo **ICL**, **LoRA** y **Full Fine-Tuning**.

In [ ]:
# 1. Verificar GPU de Colab Pro e instalar dependencias
!nvidia-smi
!pip -q install fastapi uvicorn transformers peft accelerate

In [ ]:
# 2. Subir option1/servidor_gcp/server.py a Colab (o montar Drive)
from google.colab import files
import os
if not os.path.exists("server.py"):
    print("Sube el archivo option1/servidor_gcp/server.py")
    files.upload()

In [ ]:
# 3. Arrancar FastAPI con Qwen2.5-0.5B real + túnel público Cloudflare (sin cuenta)
import subprocess, time, re, os

os.environ["RUN_REAL_GPU"] = "1"
os.environ["FM_MODEL_ID"] = "Qwen/Qwen2.5-0.5B"

!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared

uvicorn_proc = subprocess.Popen(["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

for line in tunnel_proc.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n" + "="*65)
        print("✅ PEGA ESTA URL EN option1/config.py -> GCP_API_URL:")
        print("  ", m.group(0))
        print("="*65)
        break